[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnRomanelis/Lab08/blob/main/solution/PointNet.ipynb)

# -- PointNet - Introduction to Point Cloud deep learning --

In [2]:
# If working on google colab uncomment the following line to clone the repository
# !git clone https://github.com/JohnRomanelis/Lab08.git 
# !cp -r Lab08/utils .

# Exploring the notebook (Google Colab) environment

In [1]:
import torch
torch.__version__

'2.0.0'

## Installing Libraries in Colab

In Google Colab, you can install libraries using shell commands by prefixing them with `!`. For instance, `!pip install` behaves similarly to running `pip install` in a terminal. The `-q` flag, as in `!pip install -q <package>`, performs a quiet installation, which means it suppresses the detailed output of the installation process.

In [2]:
!pip install -q torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 33.4 MB/s eta 0:00:00


In [2]:
import torch_geometric as tgm
tgm.__version__

'2.5.2'

# Data

For this exercise, we'll be working with the **ModelNet10** dataset, a collection of CAD models spanning 10 different categories.

The torch_geometric library, which we have already set up, will be used to download this dataset.

In [5]:
from torch_geometric.datasets import ModelNet
import time

t1 = time.time()
modelnet = ModelNet(root='data/ModelNet10', # Where the data will be stored
                    name='10',  # Which version of Model we want to use - ModelNet10 or ModelNet40
                    train=True  # Deep learning datasets are split in different subsets, 'training', 'evaluation', 'testing'
                    )

print(f"Time to download and process data: {(time.time() - t1)/60} min") # Should take around 8 minutes

Time to download and process data: 0.020256614685058592 min


A **Dataset** is a data structure used to manage and load data. To work correctly, a Dataset structure needs to know exactly two things: how many total items are in the dataset, and how to fetch a single data item (either loading it from disk or accessing it from memory) when given an index. For now, we will use the built-in ModelNet dataset class provided by PyTorch Geometric.

In [6]:
# total number of elements in a dataset
len(modelnet)

3991

In [7]:
# accessing a single element from the dataset
modelnet[0]

Data(pos=[7662, 3], face=[3, 12910], y=[1])

In [8]:
modelnet[0].pos[:3]

tensor([[ 6.0093,  0.0279, -7.9881],
        [ 5.8833,  0.7256, -7.9822],
        [ 5.8298,  0.6086, -8.0043]])

In [9]:
modelnet[0].face[:, :3]

tensor([[0, 3, 6],
        [1, 4, 7],
        [2, 5, 8]])

In [11]:
# Visualizing a mesh
from utils.vis import visualize_tgm_mesh
visualize_tgm_mesh(modelnet[0])

When we index the dataset, we receive a single sample representing our 3D object. This sample is an instance of PyTorch Geometric's core `Data` class. The `Data` class acts similarly to a Python dictionary, grouping all geometric attributes of a single shape into one object.

When loading a raw shape from ModelNet, the `Data` object contains three key attributes:



*   `pos` : The 3D coordinates ($X, Y, Z$) of the shape's vertices.
*   `face` : The indices that define which vertices connect to form each surface triangle.
*   `y` : The ground-truth class label (category) of the object - an intiger number.


----------------------------------------------------------------

Currently, the dataset is loading objects as **triangle meshes**. However, our goal is to implement **PointNet**, an architecture designed specifically to operate on **point clouds** rather than solid surfaces.

Thankfully, PyTorch Geometric provides a built-in sampling transform that automatically samples points uniformly across the faces of a triangle mesh. To use this feature, we pass a transform argument to the ModelNet dataset class. A `transform` is a preprocessing pipeline applied automatically to every sample right before the dataset returns it to our code.



In [12]:
import torch_geometric.transforms as T

# Sample 1024 points on the surface of each mesh
transform = T.SamplePoints(num=1024)

modelnet_pc = ModelNet(
    root='data/ModelNet10',
    name='10',
    train=True,
    transform=transform
)

In [13]:
modelnet_pc[0]

Data(pos=[1024, 3], y=[1])

In [15]:
from utils.vis import visualize_point_cloud
visualize_point_cloud(modelnet_pc[0].pos)

## Creating a custom Dataset

For learning purposes, we will now create a custom dataset, to load ModelNet10.

* First, we will use the existing dataset to store the sampled point clouds on the disk.
* Then we will our custom `Dataset` class.


In [16]:
point_clouds = []
labels = []
for item in modelnet_pc:
  pc = item.pos
  label = item.y
  point_clouds.append(pc)
  labels.append(label)


In [17]:
len(point_clouds), len(labels)

(3991, 3991)

In [18]:
import numpy as np

point_clouds = np.stack(point_clouds)
labels = np.stack(labels)

In [19]:
point_clouds.shape, labels.shape

((3991, 1024, 3), (3991, 1))

In [20]:
# Save data on the disk
np.save('data/train_point_clouds.npy', point_clouds)
np.save('data/train_labels.npy', labels)

### Custom Dataset Class

In [21]:
from torch.utils.data import Dataset


class CustomModelNet10(Dataset):

  def __init__(self, path):
    super().__init__()

    self.point_clouds = np.load(path + 'train_point_clouds.npy')
    self.labels = np.load(path + 'train_labels.npy')

  def __len__(self):
    return len(self.point_clouds)

  def __getitem__(self, index):
    pc = self.point_clouds[index]
    label = self.labels[index]

    return pc, label

In [22]:
custom_modelnet = CustomModelNet10(path='data/')

In [23]:
custom_modelnet[0]

(array([[ 10.033188 , -23.287788 ,   9.313943 ],
        [ 11.107011 ,   7.620721 ,   9.1617565],
        [  5.8558717, -27.297966 ,   6.4461765],
        ...,
        [ 10.174277 , -15.032082 ,   1.0487039],
        [  9.236888 ,  18.686316 ,  -2.9119985],
        [ 10.631059 ,   9.896687 ,   9.207667 ]], dtype=float32),
 array([0]))

In [24]:
pc, label = custom_modelnet[0]

### Transforms

To experiment even more, let's add the ability to apply custom transforms to the dataset.

In [25]:
class CustomModelNet10(Dataset):

  def __init__(self, path, transforms = []):
    super().__init__()

    self.point_clouds = np.load(path + 'train_point_clouds.npy')
    self.labels = np.load(path + 'train_labels.npy')


    # Handle single transform
    if not isinstance(transforms, (list, tuple)):
      transforms = [transforms]

    self.transforms = transforms

  def __len__(self):
    return len(self.point_clouds)

  def __getitem__(self, index):
    pc = self.point_clouds[index]
    label = self.labels[index]

    # applying transforms on the data
    for t in self.transforms:
      pc, label = t(pc, label)

    return pc, label

And now, let's create a custom transform that will change the points from a np.array to a torch.tensor.

In [26]:
class NumpyToTorch:

  def __call__(self, pc, label):
    return torch.tensor(pc), torch.tensor(label)

In [27]:
custom_modelnet = CustomModelNet10(path='data/', transforms=NumpyToTorch())

In [28]:
pc, label = custom_modelnet[0]
type(pc), type(label)

(torch.Tensor, torch.Tensor)

## Let's summarize...

Now, let's repeat the process to create a custom test set. This test set will be used to evaluate how well our model generalizes to new, unseen data. At the same time, a separate test set is essential to detect overfitting—a common failure mode where the network begins to memorize the training data by heart instead of learning generalizable geometric rules.

![image.png](https://www.freecodecamp.org/news/content/images/2023/10/overfitting-illustration.jpg)

In [29]:
## Load the dataset using ModelNet from torch geometric -- same transform as before
modelnet_pc_test = ModelNet(
    root='data/ModelNet10',
    name='10',
    train=False,
    transform=transform
)

## Save data on the disk
point_clouds = []
labels = []
for item in modelnet_pc:
  pc = item.pos
  label = item.y
  point_clouds.append(pc)
  labels.append(label)

point_clouds = np.stack(point_clouds)
labels = np.stack(labels)

np.save('data/test_point_clouds.npy', point_clouds)
np.save('data/test_labels.npy', labels)


And let's now update our custom dataset class to handle both `train` and `test` sets.

In [30]:
class CustomModelNet10(Dataset):

  def __init__(self, path, split='train', transforms = []):
    super().__init__()

    assert split in ['train', 'test'], 'Invalid split'

    self.split = split


    self.point_clouds = np.load(path + self.split + '_point_clouds.npy')
    self.labels = np.load(path + self.split + '_labels.npy')


    # Handle single transform
    if not isinstance(transforms, (list, tuple)):
      transforms = [transforms]

    self.transforms = transforms

  def __len__(self):
    return len(self.point_clouds)

  def __getitem__(self, index):
    pc = self.point_clouds[index]
    label = self.labels[index]

    # applying transforms on the data
    for t in self.transforms:
      pc, label = t(pc, label)

    return pc, label

In [31]:
train_dataset = CustomModelNet10(path='data/', split='train', transforms=NumpyToTorch())
test_dataset = CustomModelNet10(path='data/', split='test', transforms=NumpyToTorch())

## Dataloaders

Training neural networks happens in batches. This means that instead of feeding only one sample to the network at a time, we group multiple samples together into a single block of data.

1. **Computational Efficiency**: It allows the GPU to process multiple objects in parallel, maximizing hardware utilization.
2. **Training Stability**: Gradient updates calculated over a batch are less noisy than updates calculated from a single sample, leading to smoother convergence.
3. **Layer Requirements**: Essential network components like Batch Normalization explicitly require a batch dimension to calculate mean and variance statistics across multiple samples simultaneously.

PyTorch provides a built-in structure called a DataLoader that takes a dataset, automatically fetches a set of samples, and groups them together into batches for the network.

💡 `num_workers`: The DataLoader automates parallel loading by using multiple CPU cores to fetch and process data in the background. This results in much faster loading speeds, which is especially useful when handling heavy data preprocessing.

--
*For those who want to know more, we highly encourage you to look up how the collate_fn (collate function) of a PyTorch DataLoader works. It is the hidden mechanism responsible for taking a Python list of individual samples and merging them into a single, unified tensor batch!*

In [32]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4, drop_last=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4, drop_last=False)

In [33]:
train_batch = next(iter(train_dataloader))
pcs, labels = train_batch
pcs.shape, labels.shape

(torch.Size([32, 1024, 3]), torch.Size([32, 1]))

# PointNet

Let's start implementing PointNet. To do so, let's look at its architecture diagram below and the available tools PyTorch offers to build it.

In modern deep learning, libraries like PyTorch offer a great variety of fundamental building blocks. This allows us to focus on architectural design and training details rather than implementing every layer from scratch.

For now, let's skip the input transform network (T-Net) and go straight to the first shared MLP (Multi-Layer Perceptron) block. What this layer practically does is apply an MLP to each point separately. In this step, the network projects the raw 3D point coordinates $(X, Y, Z)$ into a higher-dimensional space of 64 features.

This is exactly like applying a standard fully connected (linear) layer—which you probably studied in previous courses—to every single point independently. Keep in mind that all points are fed through the exact same weights. This is conceptually equivalent to making identical copies of the linear layer for every point and running them in parallel.



![pointnet](https://stanford.edu/~rqi/pointnet/images/pointnet.jpg)

Let's see how this translates to code:


### Some torch documentation notes:

In PyTorch, neural network layers are implemented as class instances inheriting from `nn.Module`. All standard building blocks (or layers) are imported from the `torch.nn` module (conventionally aliased as `nn`).

The first layer we will look at is the nn.Linear layer. According to the [PyTorch Documentation for the Linear Layer](https://docs.pytorch.org/docs/2.12/generated/torch.nn.Linear.html), it:

*Applies an affine linear transformation to the incoming data: $ y = xA^T + b $*

💡 Notebook Power Tip: In notebook environments (like Google Colab or Jupyter), you can take a sneak peek at the documentation and internal source code of any class or function by adding a double question mark `??` before its name.
For example, to inspect the linear layer, you can run: `?? nn.Linear`

In [34]:
import torch.nn as nn

In [35]:
?? nn.Linear

Init signature:
 nn.Linear(
    in_features: int,
    out_features: int,
    bias: bool = True,
    device=None,
    dtype=None,
) -> None
Source:        
class Linear(Module):
    r"""Applies a linear transformation to the incoming data: :math:`y = xA^T + b`

    This module supports :ref:`TensorFloat32<tf32_on_ampere>`.

    On certain ROCm devices, when using float16 inputs this module will use :ref:`different precision<fp16_on_mi200>` for backward.

    Args:
        in_features: size of each input sample
        out_features: size of each output sample
        bias: If set to ``False``, the layer will not learn an additive bias.
            Default: ``True``

    Shape:
        - Input: :math:`(*, H_{in})` where :math:`*` means any number of
          dimensions including none and :math:`H_{in} = \text{in\_features}`.
        - Output: :math:`(*, H_{out})` where all but the last dimension
          are the same shape as the input and :math:`H_{out} = \text{out\_features}`.

    At

In [36]:
linear = nn.Linear(3, 16)

In [37]:
# Let's create a random point cloud
pc = torch.randn(1024, 3)
pc.shape

torch.Size([1024, 3])

In [38]:
# Let's pass the points through this linear layer
pc_proj = linear(pc)
pc_proj.shape

torch.Size([1024, 16])

At this point, it is very helpful to make a clear distinction regarding our terminology:

We have a collection of points. Each point is physically represented by its spatial $(X, Y, Z)$ coordinates. At the same time, these coordinates are the initial features of that point. As the data passes through our neural network, these features will be projected into higher dimensions, passed through activation functions, and normalized.

To understand how unique this is, let's compare point clouds directly to images:

* In **standard Images**: Pixel coordinates (the grid location row $i$, column $j$) and pixel features (the Color/RGB values) are entirely separate things. The coordinates are implicitly fixed on a rigid pixel grid, and the network learns from the color values sitting on that grid.

* In **Point Clouds**: There is no grid. The point's spatial coordinates are its initial features.

If we were working with colored point clouds, a single point's feature vector would simply be extended from 3 values to 6 values: $(X, Y, Z, R, G, B)$.

In [39]:
# Adding a non-linear function
activation = nn.ReLU()
print(pc_proj[0])
pc_proj = activation(pc_proj)
print(pc_proj[0])

tensor([-0.4263,  0.0896, -0.1093,  0.5559,  0.0705, -0.2727, -1.1744,  0.3607,
         0.5831, -0.5674, -0.1510,  0.0017,  0.0553,  0.7012,  0.5697,  0.5000],
       grad_fn=<SelectBackward0>)
tensor([0.0000, 0.0896, 0.0000, 0.5559, 0.0705, 0.0000, 0.0000, 0.3607, 0.5831,
        0.0000, 0.0000, 0.0017, 0.0553, 0.7012, 0.5697, 0.5000],
       grad_fn=<SelectBackward0>)


In [40]:
## Creating an MLP: Linear + ReLU + Linear
linear2 = nn.Linear(16, 32)

pc_proj = linear(pc)
pc_proj = activation(pc_proj)
pc_proj = linear2(pc_proj)

pc_proj.shape

torch.Size([1024, 32])

When you have a series of layers where the output of one simply passes directly into the next, you can wrap them cleanly using `nn.Sequential`.

This container bundles individual components into a single pipeline block, passing data through each layer automatically in the exact order they are defined.

In [41]:
mlp = nn.Sequential(
    nn.Linear(3, 16),
    nn.ReLU(),
    nn.Linear(16, 32)
)

pc_proj = mlp(pc)
pc_proj.shape

torch.Size([1024, 32])

## BatchNorm
In deep neural networks, since we cannot control the raw outputs of the linear layers, values can easily become unintentionally high or extremely small. These extreme values destabilize the training process.

To solve this, normalization layers were introduced. These layers use the current batch information to compute average statistics and use them to normalize the network outputs back into a stable range. This prevents values from exploding or shrinking, helping the layer outputs safely follow a standard Gaussian-like distribution.

The mathematical formulation of this operation is:$$\hat{x} = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}}$$

$$y = \gamma \ \hat{x} + \beta$$

Where $\mu$ is the batch mean, $\sigma^2$ is the batch variance, and $\gamma$ and $\beta$ are learnable parameters that allow the network to fine-tune the scale and shift of the normalized data. $\epsilon$ is a very small positive number added to the denominator to avoid division by zero.

In [42]:
mlp = nn.Sequential(
    nn.Linear(3, 16),
    nn.BatchNorm1d(16),
    nn.ReLU(),
    nn.Linear(16, 32)
)

pc_proj = mlp(pc)
pc_proj.shape

torch.Size([1024, 32])

### What about the batch dimension

We have successfully implemented an MLP using PyTorch, but how does the batch dimension affect our data structure?

So far, our tensors have been of shape $(N, F)$, where $N$ represents the number of points and $F$ represents the number of features.

However, as we discussed in the Dataloaders section, training requires us to pass a collection of multiple point clouds simultaneously during each iteration. Consequently, the tensor structure expands to include a batch dimension: $$(B, N, F)$$ Where $B$ is the batch size (the number of separate point cloud samples in the batch).

In [43]:
# Let's create a random batch of 2 point clouds
batch = torch.randn(2, 1024, 3)

In [44]:
batch_proj = linear(batch)
batch_proj.shape

torch.Size([2, 1024, 16])

In [45]:
norm = nn.BatchNorm1d(16)
batch_proj = norm(batch_proj)
batch_proj.shape

RuntimeError: running_mean should contain 1024 elements not 16

### Why this didnt work:

While a `nn.Linear` layer always expects the feature dimension to be the last dimension, PyTorch's normalization layers behave differently depending on the input shape:

* For 2D tensors $(N, F)$: `nn.BatchNorm1d` expects the features to be the last dimension. Stacking it after a linear layer works perfectly here.

* For 3D tensors $(B, N, F)$: When processing a 3D batch, `nn.BatchNorm1d` expects the features to be in the second dimension—meaning it requires a shape of $(B, F, N)$ instead of $(B, N, F)$.

To fix this, let's try permuting (swapping) these two dimensions to see if we can make it work.

In [46]:
batch_proj = batch_proj.permute(0, 2, 1)
batch_proj.shape

torch.Size([2, 16, 1024])

In [47]:
batch_proj = norm(batch_proj)
batch_proj.shape

torch.Size([2, 16, 1024])

It works!

However, doing this manual shuffling after every single layer quickly becomes messy and inefficient.

To solve this, PyTorch offers another built-in layer that natively expects 3D tensors in the $(B, F, N)$ format and pairs seamlessly with `nn.BatchNorm1d` without requiring any manual permutations: `nn.Conv1d` (with a kernel size of 1).


In [48]:
batch = batch.permute(0, 2, 1)
batch.shape

torch.Size([2, 3, 1024])

In [49]:
mlp = nn.Sequential(
    nn.Conv1d(3, 16, kernel_size=1),
    nn.BatchNorm1d(16),
    nn.ReLU(),
    nn.Conv1d(16, 32, kernel_size=1),
)

In [50]:
batch_proj = mlp(batch)
batch_proj.shape

torch.Size([2, 32, 1024])

If you take a closer look at the PointNet architecture diagram, you will notice that this combination of a **Shared MLP** (Conv1d) + **BatchNorm** + **ReLU** is the primary building block of the entire network.

Since we will be reusing this specific sequence repeatedly to project our features into higher dimensions, writing out all three layers manually every time will quickly clutter our model definition. To keep our code compact, readable, and clean, let's bundle this sequence into a custom, reusable PyTorch component!

In [51]:
class LinearLayer(nn.Module): # Always inherit nn.Module
  def __init__(self, in_features, out_features):
    super().__init__() # Always call the parent __init__

    self.layers = nn.Sequential(
        nn.Conv1d(in_features, out_features, kernel_size=1),
        nn.BatchNorm1d(out_features),
        nn.ReLU()
    )

  def forward(self, x): # This is what code is excecuted when activating the network.
    return self.layers(x)


In [52]:
linear_layer = LinearLayer(3, 16)
batch_proj = linear_layer(batch)
batch_proj.shape

torch.Size([2, 16, 1024])

## Pooling Layers

We are now missing one final component to complete the PointNet backbone architecture.

So far, we have taken an input point cloud and projected its individual points into a higher-dimensional feature space. However, our ultimate goal for a classification task is to output a single class prediction for the entire shape. This leaves us with a fundamental question: how do we transition from isolated point features to a single, unified global shape descriptor?


PointNet eleganty solves this by using a permutation-invariant aggregation function. Let's break down exactly what that means:

* **Permutation Invariant**: Point clouds are inherently unordered sets. No matter how you shuffle or permute the order of the points in your data matrix, a permutation-invariant function guarantees the exact same output.

* **Aggregation Function**: This is a mathematical operation that collapses features from all points down into a single vector. Practically, we want to eliminate the points dimension ($N$), compressing the tensor shape from $(B, F, N)$ down to $(B, F)$.
--------------------------------------------------------------

### Common Aggregation Strategies

The most common symmetric functions used in geometric deep learning are max-pooling and mean-pooling. These functions operate independently across the point dimension for every feature channel:

* **Max-Pooling**: Keeps only the maximum value for each feature across all points. For instance, if you apply a max operation directly over raw 3D coordinates, the output is a single vector containing the maximum $X$, maximum $Y$, and maximum $Z$ values found across the entire cloud.

* **Mean-Pooling**: Computes the average value for each feature. Applying this to raw coordinates is conceptually identical to finding the physical center of mass of the point cloud.

In [53]:
## Let's create a single vector for each point cloud in the batch
# batch_proj is of shape (B, F, N) --> we apply the reduction over the last dimension
batch_vector = batch_proj.max(dim=2)[0]
batch_vector.shape

torch.Size([2, 16])

Notice that now, after the aggregation function has eliminated the point dimension, the resulting vector is 2-dimensional with a shape of $(B, F)$.

Because we are back to a standard 2D matrix layout, when we pass this global signature through the final classification MLP, we can cleanly use standard `nn.Linear` layers paired directly with `nn.BatchNorm1d` without running into any of the dimension mismatch errors we encountered earlier!

In [54]:
cls_mlp = nn.Sequential(
    nn.Linear(16, 32),
    nn.BatchNorm1d(32),
    nn.ReLU(),
    nn.Linear(32, 10) # 10 is the number of classes
)

out = cls_mlp(batch_vector)
out.shape

torch.Size([2, 10])

### Simple PointNet

Now that we have successfully designed and tested all the necessary components, we can go ahead and combine them to create a complete, working Simple PointNet architecture!

In [55]:
class SimplePointNet(nn.Module):
  def __init__(self, num_classes):
    super().__init__()

    self.mlp1 = nn.Sequential(
        LinearLayer(3, 64),
        LinearLayer(64, 64)
    )

    self.mlp2 = nn.Sequential(
        LinearLayer(64, 64),
        LinearLayer(64, 128),
        LinearLayer(128, 1024)
    )


    self.cls_head = nn.Sequential(
        nn.Linear(1024, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(),
        nn.Linear(512, 256),
        nn.BatchNorm1d(256),
        nn.ReLU(),
        nn.Dropout(0.3), # --> Food for thought...
        nn.Linear(256, num_classes)
    )


  def forward(self, x):

    x = self.mlp1(x)

    x = self.mlp2(x)

    # Aggregation
    x_vector = x.max(dim=-1)[0]

    out = self.cls_head(x_vector)

    return out

In [56]:
model = SimplePointNet(num_classes=10)

In [57]:
prediction = model(batch)
prediction.shape

torch.Size([2, 10])

# Training the network

Now that we have the data and we have the model, what is left to do is use this data to actually train the network. To do this, we need to answer two fundamental questions:

1. How do we measure how well our model is doing?

2. How do we update the network to make it better?


## The Training Objective: The Loss Function

Before we can improve our network, we need a target for the model to minimize. This target is called the Loss Function.

Think of the loss function as a referee or a scorekeeper. It takes the model’s predictions, compares them directly to the real ground-truth labels, and calculates a single number (a penalty score) that tells us exactly how wrong the model is. If the model makes a perfect guess, the loss is very low. If it makes a terrible guess, the loss is high. Our ultimate goal during training is to get this loss value as close to zero as possible.

------------------------------------------------------
### Theoretical Background
------------------------------------------------------
### Understanding the Output and the Loss Function

However, before we can compute a loss, we need to understand exactly what our network outputs.

For a classification task with 10 categories, our Simple PointNet outputs a vector of 10 values for each point cloud—one score for each possible class. The value of each number indicates the network's confidence that the point cloud belongs to that specific category. Therefore, whichever "neuron" has the highest activation value corresponds to the final class we will assign to that model.

### L1 or L2 loss

We could then use a basic distance formula, like L1 loss (absolute difference) or L2 loss (squared difference), to calculate how far our network's raw scores are from this perfect 1-and-0 target. While this makes intuitive sense, it struggles in practice because raw network scores can scale infinitely, making training unstable. While you could use a Sigmoid function here to squash each output between 0 and 1, it treats every class as an independent yes/no question rather than forcing the network to choose the single most likely category out of the 10.

### Cross-Entropy loss

Instead of dealing with raw, unpredictable activation scores, Cross-Entropy operates on probabilities. To do this, it first maps the raw network outputs ($z$) into a clean probability distribution using the Softmax formula:

$$P(class\_i) = \frac{e^{z_i}}{\sum_{j=1}^{10} e^{z_j}}$$

This exponential mapping guarantees that every class gets a probability between 0 and 1, and that the sum of all 10 probabilities equals exactly $1.0$ ($100\%$).

Once our outputs are mapped to clean probabilities, Cross-Entropy calculates the final error penalty using this formula:

$$Loss = -\log(P(correct\_class))$$

Looking at this math, our task for the network becomes very clear: we want it to produce a probability of exactly 1 for the correct neuron. If the network achieves $P = 1$ for the correct class, $-\log(1)$ equals $0$, resulting in zero loss. If the network is unconfident and outputs a small probability near $0$, the $-\log$ penalty spikes toward infinity, forcing the model to aggressively adjust its weights.

The standard text-book equation for cross-entropy loss covering a single sample with $C$ classes is written as:
$$Loss = -\sum_{c=1}^{C} y_c \log(p_c)$$
Where:
* $y_c$ is the ground-truth target for class $c$ (which is $1$ for the correct class and $0$ for all others).
* $p_c$ is the predicted probability for class $c$.

Because $y_c$ is equal to zero for every single incorrect class, all of those terms drop out of the summation completely.


In [58]:
# Defining the loss function - aka criterion
criterion = nn.CrossEntropyLoss()

In [59]:
# Creating some fake labels for our dummy point cloud
dummy_labels = torch.tensor([0, 2])

In [60]:
loss = criterion(prediction, dummy_labels)
loss

tensor(2.6427, grad_fn=<NllLossBackward0>)

## Updating the Weights:

----------------------------------------------------------------
### 1. Gradient Computation
----------------------------------------------------------------
Before we can adjust the internal weights (the settings and dials) of our network, we need to know how changing them will affect the total loss. This requires calculating a partial derivative for every single weight ($w$) in the network:

$$\text{Gradient} = \frac{\partial \text{Loss}}{\partial w}$$

This partial derivative answers a simple question: "If I turn this specific weight dial up a tiny bit, how much will the loss change?"

The process of calculating all of these partial derivatives across the entire network is called **Backpropagation**. It is called "backpropagation" because the algorithm starts at the very end of the network (where the loss score is calculated) and works its way backward.

In traditional calculus, calculating these derivatives requires **Analytic Differentiation**—meaning you have to manually derive a giant, complex mathematical formula on paper for the entire network's derivative and then hardcode it. If you change your model architecture even slightly, your formulas break, and you have to start over.

PyTorch solves this completely using **Automatic Differentiation** (via its `autograd` engine). Instead of calculating one massive formula, PyTorch breaks your network down into a sequence of basic operations (addition, multiplication, etc.). As your data flows forward through the network, PyTorch builds a dynamic graph of these tiny operations. During backpropagation, it applies the calculus Chain Rule to multiply the derivatives of these small steps together automatically:

$$\frac{\partial \text{Loss}}{\partial w} = \frac{\partial \text{Loss}}{\partial \text{Output}} \times \frac{\partial \text{Output}}{\partial w}$$

This gives us the exact same mathematically perfect gradient vector as manual calculus, but with zero manual math required from us!



In [61]:
# To compute the gradients simply run
loss.backward()

In [62]:
# we can manually access the gradient values of the network weights
model.mlp1[0].layers[0].weight.grad.shape

torch.Size([64, 3, 1])

----------------------------------------------------------------
### 2. Updating the weights
----------------------------------------------------------------
Once backpropagation finishes calculating our directional gradients, an algorithm called an Optimizer takes over to actually update the weights.

Think of your neural network as a person trying to walk down a foggy mountain (the loss) to reach the lowest possible valley (zero loss). The gradients tell the person which way is straight uphill. Because we want to go down the mountain, we take a step in the exact opposite direction of the gradient.

The most fundamental optimizer that does this is Stochastic Gradient Descent (SGD). Its mathematical update rule is incredibly straightforward:

$$w_{\text{new}} = w_{\text{old}} - \eta \times \frac{\partial \text{Loss}}{\partial w}$$

Where $\eta$ is our **Learning Rate**, a small number (like $0.01$ or $0.001$) that acts as our step size. If our step size is too big, we might accidentally leap across the canyon and miss the valley completely. If it is too small, it will take the person an eternity to get down the mountain.

SGD scales our steps based on the gradient: when the mountain is steep, the gradient is large and we take bigger steps; as we approach the flat valley floor, the gradient shrinks to near zero, naturally slowing our updates down.

In [63]:
import torch.optim as optim

In [64]:
optimizer = optim.SGD(model.parameters(), lr=0.01)

In [65]:
# to update the weight values:
optimizer.step()

#### 🚨 Important: Resetting Gradients

Before computing new gradients, you must clear out any old gradients left over from previous batches.

Because PyTorch accumulates (adds) new gradients to existing ones instead of overwriting them, failing to reset them will mathematically corrupt your training updates.

So before any gradient computation you should run `optimizer.zero_grad()`


In [66]:
optimizer.zero_grad() # --> Deletes the gradient!
model.mlp1[0].layers[0].weight.grad == None

True

## Training Loop

In [67]:
from tqdm import tqdm # progress bar

In [68]:
epochs = 1
lr = 0.01

model = SimplePointNet(num_classes=10)
optimizer = optim.SGD(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()

for epoch in range(epochs):
  # ============================
  #       TRAINING PHASE
  # ============================
  model.train() # <-- Important for layers like BatchNorm

  running_loss = 0.0  # To compute the average loss per epoch
  correct_train = 0   # Correctly classified samples
  total_train = 0     # Total samples



  for batch in tqdm(train_dataloader):
    x, y = batch
    x = x.permute(0, 2, 1) # x.shape (B, N, F) --> (B, F, N)
    y = y.squeeze() # y.shape (B, 1) --> (B,)
    optimizer.zero_grad()
    prediction = model(x)
    loss = criterion(prediction, y)
    loss.backward()
    optimizer.step()

    running_loss += loss.item() * x.shape[0] # loss is averaged across the batch

    _, predicted_classes = torch.max(prediction, 1)
    total_train += x.shape[0]
    correct_train += (predicted_classes == y).sum().item()

  epoch_train_loss = running_loss / total_train
  epoch_train_acc = correct_train / total_train * 100
  print(f"-> [Summary] Epoch {epoch+1} | Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.2f}%")

  # ============================
  #       VALIDATION PHASE
  # ============================
  model.eval()

  running_val_loss = 0.0
  correct_val = 0
  total_val = 0

  with torch.no_grad():
    for batch in tqdm(test_dataloader):
      x, y = batch
      x = x.permute(0, 2, 1)
      y = y.squeeze()
      prediction = model(x)
      val_loss = criterion(prediction, y)

      # --- Metrics Tracking ---
      running_val_loss += val_loss.item() * x.size(0)
      _, predicted_classes = torch.max(prediction, dim=1)
      correct_val += (predicted_classes == y).sum().item()
      total_val += y.size(0)

    epoch_val_loss = running_val_loss / total_val
    epoch_val_acc = (correct_val / total_val) * 100

    print(f"   ✨ [Validation Results] Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.2f}%")
    print("-" * 60)

  0%|          | 0/124 [00:00<?, ?it/s]

100%|██████████| 124/124 [00:45<00:00,  2.73it/s]


-> [Summary] Epoch 1 | Train Loss: 1.8601 | Train Acc: 39.87%


100%|██████████| 125/125 [00:16<00:00,  7.56it/s]

   ✨ [Validation Results] Val Loss: 2.4864 | Val Acc: 13.03%
------------------------------------------------------------


## Running on the GPU!

In [70]:
epochs = 1
lr = 0.01

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print("Using device: ", device)

model = SimplePointNet(num_classes=10).to(device)
optimizer = optim.SGD(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()

for epoch in range(epochs):
  # ============================
  #       TRAINING PHASE
  # ============================
  model.train() # <-- Important for layers like BatchNorm

  running_loss = 0.0  # To compute the average loss per epoch
  correct_train = 0   # Correctly classified samples
  total_train = 0     # Total samples



  for batch in tqdm(train_dataloader):
    x, y = batch
    x = x.to(device)
    y = y.to(device)

    x = x.permute(0, 2, 1) # x.shape (B, N, F) --> (B, F, N)
    y = y.squeeze() # y.shape (B, 1) --> (B,)
    optimizer.zero_grad()
    prediction = model(x)
    loss = criterion(prediction, y)
    loss.backward()
    optimizer.step()

    running_loss += loss.item() * x.shape[0] # loss is averaged across the batch

    _, predicted_classes = torch.max(prediction, 1)
    total_train += x.shape[0]
    correct_train += (predicted_classes == y).sum().item()

  epoch_train_loss = running_loss / total_train
  epoch_train_acc = correct_train / total_train * 100
  print(f"-> [Summary] Epoch {epoch+1} | Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.2f}%")

  # ============================
  #       VALIDATION PHASE
  # ============================
  model.eval()

  running_val_loss = 0.0
  correct_val = 0
  total_val = 0

  with torch.no_grad():
    for batch in tqdm(test_dataloader):
      x, y = batch
      x = x.to(device)
      y = y.to(device)

      x = x.permute(0, 2, 1)
      y = y.squeeze()

      prediction = model(x)
      val_loss = criterion(prediction, y)

      # --- Metrics Tracking ---
      running_val_loss += val_loss.item() * x.size(0)
      _, predicted_classes = torch.max(prediction, dim=1)
      correct_val += (predicted_classes == y).sum().item()
      total_val += y.size(0)

    epoch_val_loss = running_val_loss / total_val
    epoch_val_acc = (correct_val / total_val) * 100

    print(f"   ✨ [Validation Results] Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.2f}%")
    print("-" * 60)

Using device:  cuda


100%|██████████| 124/124 [00:02<00:00, 57.96it/s] 


-> [Summary] Epoch 1 | Train Loss: 1.9586 | Train Acc: 37.40%


100%|██████████| 125/125 [00:00<00:00, 158.20it/s]

   ✨ [Validation Results] Val Loss: 2.2401 | Val Acc: 23.75%
------------------------------------------------------------


# Data Proprocessing

Data preprocessing is one of the most important steps in deep learning. This step includes processing the data so that they are in the correct form, allowing the network to learn efficiently.

To prove our point, let's start by checking the scale of different examples in our dataset.

In [71]:
pc1 = custom_modelnet[0][0]
pc1.min(), pc1.max()

(tensor(-33.7590), tensor(33.9300))

In [72]:
pc2 = custom_modelnet[-1][0]
pc2.min(), pc2.max()

(tensor(-20.0061), tensor(19.9196))

Here we can clearly see that the range of input values varies significantly between different samples. This introduces two major types of dangers:



1.   `Optimization Difficulty`: The network architecture will have a hard time processing these data, as it is proven neural networks work best when input features are normalized to a consistent scale (typically centered around 0 with a small variance ie 1). High variation in input scale leads to unstable training and usually to suboptimal results.
2.   `Data Leakage`: If different classes have different scales, it is possible that this scale information leaks class information. We want to train a network that classifies point clouds based purely on their structural geometry, not simply because a "bed" happens to be physically larger than a "chair" in the data files.

In [73]:
# Create a unit sphere normalization transform

class UnitSphereNormalize:

  def __call__(self, x, y):
    # x.shape N, 3
    x = x-x.mean(dim=0, keepdim=True)

    radius = torch.sqrt(torch.sum(x**2, dim=1, keepdim=True))
    print(radius.shape)
    x = x/radius

    return x, y

In [74]:
train_dataset = CustomModelNet10(path='data/', split='train', transforms=[NumpyToTorch(), UnitSphereNormalize()])
test_dataset = CustomModelNet10(path='data/', split='test', transforms=[NumpyToTorch(), UnitSphereNormalize()])

In [75]:
pc1 = train_dataset[0][0]
pc1.min(), pc1.max()

torch.Size([1024, 1])


(tensor(-0.9988), tensor(1.0000))

In [76]:
pc2 = test_dataset[-1][0]
pc2.min(), pc2.max()

torch.Size([1024, 1])


(tensor(-0.9995), tensor(0.9994))

### Train with normalized data

In [79]:
epochs = 1
lr = 0.01

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print("Using device: ", device)

model = SimplePointNet(num_classes=10).to(device)
optimizer = optim.SGD(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()

for epoch in range(epochs):
  # ============================
  #       TRAINING PHASE
  # ============================
  model.train() # <-- Important for layers like BatchNorm

  running_loss = 0.0  # To compute the average loss per epoch
  correct_train = 0   # Correctly classified samples
  total_train = 0     # Total samples



  for batch in tqdm(train_dataloader):
    x, y = batch
    x = x.to(device)
    y = y.to(device)

    x = x.permute(0, 2, 1) # x.shape (B, N, F) --> (B, F, N)
    y = y.squeeze() # y.shape (B, 1) --> (B,)
    optimizer.zero_grad()
    prediction = model(x)
    loss = criterion(prediction, y)
    loss.backward()
    optimizer.step()

    running_loss += loss.item() * x.shape[0] # loss is averaged across the batch

    _, predicted_classes = torch.max(prediction, 1)
    total_train += x.shape[0]
    correct_train += (predicted_classes == y).sum().item()

  epoch_train_loss = running_loss / total_train
  epoch_train_acc = correct_train / total_train * 100
  print(f"-> [Summary] Epoch {epoch+1} | Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.2f}%")

  # ============================
  #       VALIDATION PHASE
  # ============================
  model.eval()

  running_val_loss = 0.0
  correct_val = 0
  total_val = 0

  with torch.no_grad():
    for batch in tqdm(test_dataloader):
      x, y = batch
      x = x.to(device)
      y = y.to(device)

      x = x.permute(0, 2, 1)
      y = y.squeeze()

      prediction = model(x)
      val_loss = criterion(prediction, y)

      # --- Metrics Tracking ---
      running_val_loss += val_loss.item() * x.size(0)
      _, predicted_classes = torch.max(prediction, dim=1)
      correct_val += (predicted_classes == y).sum().item()
      total_val += y.size(0)

    epoch_val_loss = running_val_loss / total_val
    epoch_val_acc = (correct_val / total_val) * 100

    print(f"   ✨ [Validation Results] Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.2f}%")
    print("-" * 60)

Using device:  cuda


  0%|          | 0/124 [00:00<?, ?it/s]

100%|██████████| 124/124 [00:01<00:00, 100.20it/s]


-> [Summary] Epoch 1 | Train Loss: 1.9344 | Train Acc: 37.47%


100%|██████████| 125/125 [00:00<00:00, 155.59it/s]

   ✨ [Validation Results] Val Loss: 2.3463 | Val Acc: 24.56%
------------------------------------------------------------
